# Notebook — Fonctionnalités

Outils annexes du pipeline (dataset, regroup, sanity checks, exports, vidéo).
Les 3 phases de génération (generate / render / evaluate) sont dans
`notebook_generation.ipynb`.

## 1. Setup

**À exécuter en premier.** Localise la racine du projet, ajoute `project/` au
`sys.path`, puis importe les fonctions outils (`build_dataset`, `regroup_images`,
`build_yolo_box`, `build_lard_box`, `build_video`, `find_runs`, etc.). Les 3
phases de génération (`generate_runs`, `render_runs`, `evaluate_runs`) sont dans
`notebook_generation.ipynb`. Affiche `ROOT` et `RUNS_DIR` pour vérifier que les
chemins sont corrects.

In [1]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "run_pipeline.py").exists():
            return p
    return start

ROOT = _find_root(Path.cwd())

_project = str(ROOT / "project")
if _project not in sys.path:
    sys.path.insert(0, _project)

from notebook_tools import (
    RUNS_DIR,
    build_dataset, regroup_images,
    build_yolo_box, show_sanity,
    build_lard_box, show_sanity_lard,
    build_xplane_config, build_params_trace,
    build_video,
)
from runs import find_runs  # utilisé dans la cellule "Lister les runs"

print(f"ROOT     = {ROOT}")
print(f"RUNS_DIR = {RUNS_DIR}  (exists={RUNS_DIR.exists()})")

ROOT     = c:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF
RUNS_DIR = C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs  (exists=True)


## 2. Lister les runs disponibles

Énumère tous les runs valides présents dans `runs/` (toutes générations
confondues), affichés en chemin relatif `<generation>/<run>`. Pratique pour
récupérer le nom exact à passer aux fonctions des sections suivantes.

In [5]:
runs = find_runs(all_runs=True)
for r in runs:
    print(f"  - {r.relative_to(RUNS_DIR)}")
print(f"\n{len(runs)} run(s) trouve(s).")

  - generation_01\KIND_32
  - generation_02\LFPO_25
  - generation_02\OLBA_17
  - generation_02\SBGR_10L

4 run(s) trouve(s).


## 3. Création du dataset (arborescence par piste / scénario)

Format aligné sur le CSV LARD natif (sous-ensemble : colonnes pertinentes pour
notre pipeline). Les valeurs `yaw`, `pitch`, `roll`, `slant_distance`,
`along_track_distance`, `height_above_runway`, et la bbox `x_TR..y_BR` sont
copiées telles quelles depuis les *_labels.csv (unités LARD : degrés, NM, ft, px).

```
dataset/
├── metadata.csv                         # toutes pistes, tous scénarios
└── <ICAO_RWY>/                          # ex: KPDX_10L
    ├── metadata.csv                     # tous scénarios de cette piste
    └── <ICAO_RWY>_<NNN>/                # ex: KPDX_10L_001
        ├── metadata.csv                 # ce scénario uniquement
        └── images/000000.jpg ...
```

Colonnes : `height, width, type, scenario, airport, runway, time, time_fps,
weather, yaw, pitch, roll, slant_distance, along_track_distance,
height_above_runway, lateral_path_angle, vertical_path_angle,
x_TR, y_TR, x_TL, y_TL, x_BL, y_BL, x_BR, y_BR, image`.

Nécessite que la génération ait été faite avec le nouveau Export.py (ajoute
`template_file_name` dans `poses_cam_export.json` -> colonne `weather`).

In [6]:
build_dataset()

Dataset : C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\dataset
  KIND_32                   <- footage  (   0 imgs) [absent]
  LFPO_25                   <- footage  ( 270 imgs) [ok -> LFPO_25/LFPO_25_001]
  OLBA_17                   <- footage  (   0 imgs) [absent]
  SBGR_10L                  <- footage  (   0 imgs) [absent]
Total : 270 images, 4 piste(s)
Metadata racine : C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\dataset\metadata.csv (270 lignes)


{'KIND_32': ('footage', 0, 'absent'),
 'LFPO_25': ('footage', 270, 'ok -> LFPO_25/LFPO_25_001'),
 'OLBA_17': ('footage', 0, 'absent'),
 'SBGR_10L': ('footage', 0, 'absent')}

In [ ]:
# regroup_images() — regroupe + renumerote les images du dataset, avec metadata.csv.
#
# mode="piste" (defaut) : un dossier par piste, renumerotation par piste.
#     runs/dataset_regroup/<RWY>/img/000000.jpg ...
#     runs/dataset_regroup/<RWY>/metadata.csv      (scenarios de cette piste)
#
# mode="all"            : tout dans un seul dossier, renumerotation globale.
#     runs/dataset_regroup/datasetr/img/000000.jpg ...
#     runs/dataset_regroup/datasetr/metadata.csv   (toutes pistes confondues)
#
# Les deux modes coexistent (chacun ne nettoie que son propre sous-dossier).

regroup_images(mode="piste")
# regroup_images(mode="all")

Regroupement global : C:\Users\Sr-Sh\Desktop\LARD-LAAS-TAF\runs\dataset_regroup\datasetr
  LFPO_25                    270 imgs
Total : 270 images, 1 piste(s)


## 4. Générer `yolo_box/` (images avec bbox YOLO dessinées)

Lance l'inférence YOLO sur les images sources (`degraded/` en priorité, sinon
`footage/`) et sauve les images annotées dans `runs/<gen>/<run>/yolo_box/`.

Usage :
- `build_yolo_box()`                          : toutes les runs
- `build_yolo_box("KLAX_25R")`                : un seul run (si unique)
- `build_yolo_box("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_yolo_box("generation_01/KLAX_25R")  # un seul run
build_yolo_box()

## 5. Sanity check : 3 images avec bbox YOLO 

Affiche première / milieu / dernière image d'UN run avec les bbox YOLO dessinées
(`degraded/` prio sinon `footage/`). Pas de mode "tous les runs" : sans argument,
prend le premier run trouvé.

Usage :
- `show_sanity()`                          : premier run trouvé
- `show_sanity("KLAX_25R")`                : un run cible (si unique)
- `show_sanity("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# show_sanity("generation_01/KLAX_25R")  # cible un run spécifique
show_sanity()

## 6. Générer `lard_box/` (images avec bbox GT LARD dessinées)

Usage :
- `build_lard_box()`                          : toutes les runs
- `build_lard_box("KLAX_25R")`                : un seul run (si unique)
- `build_lard_box("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_lard_box("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_lard_box()

## 7. Sanity check : 3 images avec bbox GT LARD 

Affiche première / milieu / dernière image d'UN run avec les 4 coins GT LARD
(`degraded/` prio sinon `footage/`). Pas de mode "tous les runs" : sans argument,
prend le premier run trouvé.

Usage :
- `show_sanity_lard()`                          : premier run trouvé
- `show_sanity_lard("KLAX_25R")`                : un run cible (si unique)
- `show_sanity_lard("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# show_sanity_lard("generation_01/KLAX_25R")  # cible un run spécifique
show_sanity_lard()

## 8. Générer `xplane_config.json`

Reconstruit `runs/<gen>/<run>/xplane_config.json` depuis le yaml (résolution pré-crop,
FOV) et `weather_profile.json` (status). `pilot_eye_*` dépend de l'avion choisi
sur X-Plane 12.

**Contenu** : `width`/`height` (résolution), `fov_h`/`fov_v` (champ de vision),
`pilot_eye_x/y/z` (position œil pilote), `weather_status` (ok/absent).

Usage :
- `build_xplane_config()`                          : toutes les runs
- `build_xplane_config("KLAX_25R")`                : un seul run (si unique)
- `build_xplane_config("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_xplane_config("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_xplane_config()

## 9. Générer `params_trace.xml`

Reconstruit `runs/<gen>/<run>/params_trace.xml` en agrégeant `poses_cam_export.json`,
`weather_profile.json` et `fault_profile.json`.

**Contenu** : un XML `test_case` > `scenario` avec 3 blocs — `trajectory`
(fps + paramètres de trajectoire), `weather` (paramètres météo), `faults`
(fautes capteur appliquées).

Usage :
- `build_params_trace()`                          : toutes les runs
- `build_params_trace("KLAX_25R")`                : un seul run (si unique)
- `build_params_trace("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_params_trace("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_params_trace()

## 10. Générer une vidéo MP4 d'un run/des runs

Concatène les images du run (`degraded/` en priorité sinon `footage/`) en un MP4 (fps du yaml).

Usage :
- `build_video()`                          : toutes les runs
- `build_video("generation_01/KLAX_25R")`  : chemin composé
- `build_video(source="footage")`          : force la source (`footage` ou `degraded`)

In [ ]:
# build_video("generation_02/LFPO_25", source="footage")  # run spécifique + source footage
build_video() 